In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

In [3]:
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    #extra = 0
    #if np.min(x) < 0:
        #extra = np.min(x)
        #x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f)#+extra)
    return(prob.value,q.value,q_b.value)

In [4]:
def dual (sets,p,R,r,m,r_f,a):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R.dot(a))[i]-(1-sum(a))*r_f - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    obj= cp.Minimize(alpha + beta + gamma * (r-1) + z4 + z2)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,v.value,lbda.value,alpha.value,beta.value,gamma.value,t.value)

In [5]:
np.random.seed(10)

In [9]:
N=10
p = (np.zeros(N)+1)*1/N
I = 1
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
print(R)

[0.00368284]
[[ 0.07349513]
 [-0.33149138]
 [-0.13458185]
 [ 0.14395029]
 [ 0.02112665]
 [-0.03002767]
 [-0.00919677]
 [ 0.21964172]
 [ 0.19136609]
 [-0.10745379]]


In [10]:
a = np.zeros(I)+1/I
r = 0.043
m = 0.95
r_f = 0.001
robustcheck(a,R,r,p,m,r_f)

0.33149137762481423


(0.33149137762481423,
 array([0.09868711, 0.10530076, 0.10084778, 0.09894536, 0.0985908 ,
        0.09911526, 0.09874553, 0.10048883, 0.09948331, 0.09979526]),
 array([7.05726641e-14, 1.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        3.17177078e-14, 0.00000000e+00, 0.00000000e+00, 4.48626952e-13,
        0.00000000e+00, 0.00000000e+00]))

In [11]:
        
x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
sets =psets

In [12]:
dual (sets,p,R,r,m,r_f,a)

(0.33149137760261393,
 array([[3.97158965e-14, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 5.52330834e-14, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 3.98080868e-14, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [2.82359059e-14, 0.00000000e+00, 3.01785260e-14, ...,
         2.85570729e-14, 2.91836765e-14, 3.08453851e-14],
        [0.00000000e+00, 2.44743111e-14, 3.03198775e-14, ...,
         2.86838517e-14, 2.93159237e-14, 3.09920014e-14],
        [2.81779578e-14, 2.43312732e-14, 3.01186141e-14, ...,
         2.84989026e-14, 2.91247207e-14, 3.07844430e-14]]),
 array([7.39489495e-15, 3.31573238e-04, 6.11059296e-15, ...,
        1.49444841e-14, 1.80343861e-02, 8.27962844e-02]),
 array([1.91081986e-11]),
 array([-0.65243649]),
 array([1.80352056e-12]),
 array([1.61552064e-12, 2.13999666e-12, 2.2435060

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]
